# Session 02 Lab: RAG Pipeline Step-by-Step with Jupyter & Gemini

ในแล็บนี้ เราจะมาสร้างระบบ **Retrieval-Augmented Generation (RAG)** แบบง่ายๆ ด้วยตนเองโดยยังไม่ใช้ Airflow โดยเราจะทำตามกระบวนการหลักของ RAG ทั้งหมด:
1. **Document Preparation**: การสกัดเนื้อหาจากเอกสาร PDF
2. **Chunking**: การแบ่งเอกสารเป็นขนาดเล็กๆ
3. **Embedding**: การแปลงรูปคำพูดตัวอักษรเป็นเวกเตอร์โดยโมเดลของ Gemini
4. **Vector Database**: การสร้างและจัดเก็บเวกเตอร์ลง ChromaDB
5. **Retrieval**: การคิวรีสืบค้นข้อมูลที่คล้ายกันออกมา
6. **Augmented Generation**: การสร้าง Prompt และส่งหา Gemini LLM เพื่อตอบคำถามผู้ใช้

## 1. ติดตั้งไลบรารีและตั้งค่า API Key

หากคุณรันผ่าน Docker-Compose ระบบได้ลงแพ็คเกจไว้ให้เรียบร้อยแล้ว ในกรณีอื่น ให้รันคำสั่งด้านล่างเพื่อลงไลบรารีที่เกี่ยวข้อง

In [ ]:
# !pip install google-genai chromadb langchain-text-splitters PyMuPDF

In [ ]:
import os
from google import genai
from google.genai import types
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ดึง API Key จากตัวแปรระบบ (หรือคุณสามารถระบุตรงๆ ได้ชั่วคราวเพื่อรันศึกษา)
gemini_api_key = os.environ.get("GEMINI_API_KEY", "")

if not gemini_api_key:
    client = None
    print("กรุณากรอก GEMINI_API_KEY เพื่อเปิดสิทธิเรียกใช้บริการ Google AI Studio")
else:
    client = genai.Client(api_key=gemini_api_key)
    print("เชื่อมต่อบริการ Google Gen AI สำเร็จ!")

## 2. การจัดเตรียมเอกสารและการสกัดข้อความจาก PDF (Document Preparation)

เราจะเปิดไฟล์ PDF นโยบายการทำงาน WFH (`policy_wfh.pdf`) และสกัดเนื้อหาทั้งหมดออกมาเป็นข้อความ (Text) โดยใช้ไลบรารี PyMuPDF (`fitz`)

In [ ]:
import fitz  # PyMuPDF สำหรับดึงเนื้อหาจาก PDF

pdf_path = "../documents/policy_wfh.pdf"
doc = fitz.open(pdf_path)
sample_document = ""
for page in doc:
    sample_document += page.get_text()

print("สกัดเนื้อหาจาก PDF สำเร็จ!")
print("ขนาดเอกสารตัวอย่าง:", len(sample_document), "ตัวอักษร")
print("\n--- ตัวอย่างเนื้อหาในไฟล์ ---\n")
print(sample_document)

## 3. การแบ่งข้อความเป็นขนาดเล็ก (Chunking)

หากป้อนเอกสารยาวทั้งหมดเข้า Prompt จะสิ้นเปลืองค่า Token และทำให้โมเดลสับสนในการค้นหาข้อมูล เราจึงใช้ `RecursiveCharacterTextSplitter` เพื่อแบ่งเอกสารเป็นตอนเล็กๆ โดยให้มีความเหลื่อมล้ำกันเล็กน้อย (Overlap) เพื่อไม่ให้ความหมายของขอบรอยต่อหลุดหายไป

In [ ]:
# กำหนดขนาด Chunk
chunk_size = 250
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(sample_document)

print(f"แบ่งเอกสารออกได้เป็น {len(chunks)} ชิ้น (Chunks):")
for i, chunk in enumerate(chunks):
    print(f"\n[Chunk {i+1}]: Size: {len(chunk)}\n{chunk}\n{'-'*40}")

## 4. การทำ Embedding ด้วยโมเดล Gemini

เราจะเขียนฟังก์ชันสำหรับแปลงข้อความ (Text) ให้กลายเป็นค่าเวกเตอร์ (List of Floats) โดยเรียกใช้โมเดล `models/text-embedding-004` ของ Google AI Studio

In [ ]:
def get_gemini_embedding(text):
    try:
        result = client.models.embed_content(
            model="models/gemini-embedding-2",
            contents=text,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        return result.embeddings[0].values
    except Exception as e:
        print(f"เกิดข้อผิดพลาดในการ Embed: {e}")
        return []

# ทดลองสร้างเวกเตอร์จากตัวอย่างแรก
test_vector = get_gemini_embedding(chunks[0])
print(f"ขนาดมิติ (Dimension) ของ Vector Embedding: {len(test_vector)}")
print("ตัวอย่างเวกเตอร์ 5 มิติแรก:", test_vector[:5])

## 5. การจัดเก็บเวกเตอร์ลงในฐานข้อมูล ChromaDB

เราจะเปิดใช้งาน ChromaDB แบบในหน่วยความจำ (In-memory) หรือ Local Storage และสร้าง Collection เก็บข้อมูล Chunks รวมถึง เวกเตอร์ที่เราคำนวณไว้ข้างต้น

In [ ]:
# 1. สร้าง Chroma Client แบบฝังตัว (Local/Ephemeral)
chroma_client = chromadb.EphemeralClient()

# 2. สร้าง Collection ในฐานข้อมูล
collection_name = "kx_welfare_benefits"

# ลบ Collection เดิมหากซ้ำซ้อนเพื่อให้ระบบรันซ้ำได้โดยไม่มีข้อมูลเบิ้ล (Idempotency)
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)

# 3. ทำลูปนำเข้าข้อมูล Chunks
ids = []
embeddings = []
documents = []
metadatas = []

for i, chunk in enumerate(chunks):
    chunk_id = f"chunk_{i}"
    vector = get_gemini_embedding(chunk)
    
    ids.append(chunk_id)
    embeddings.append(vector)
    documents.append(chunk)
    metadatas.append({"source": "policy_wfh.pdf", "chunk_index": i})

# บันทึกข้อมูลเข้า ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print(f"เพิ่มเอกสารจำนวน {collection.count()} Chunks เข้าสู่ ChromaDB เรียบร้อยแล้ว!")

## 6. การสืบค้นเนื้อหาที่คล้ายคลึงกัน (Retrieval)

เมื่อผู้ถามป้อนคำถามเข้ามา เราจะทำการแปลงคำถามนั้นเป็นเวกเตอร์ความรู้ก่อน แล้วไปค้นหา Chunk ในฐานข้อมูล ChromaDB ที่มีค่าเวกเตอร์ใกล้เคียงคำถามที่สุด

In [ ]:
query_text = "ฉันทำงานจากที่บ้าน (WFH) ได้สูงสุดกี่วันต่อสัปดาห์ และต้องได้รับการอนุมัติในวันใด?"

# 1. แปลงคำถามผู้ถามเป็นเวกเตอร์
query_vector = client.models.embed_content(
    model="models/gemini-embedding-2",
    contents=query_text,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
).embeddings[0].values

# 2. ค้นหาใน ChromaDB (ดึงมา 2 อันที่คะแนนใกล้เคียงสุด)
results = collection.query(
    query_embeddings=[query_vector],
    n_results=2
)

print(f"คำถาม: {query_text}\n")
print("ผลการดึงข้อความบริบท:")
for i, doc in enumerate(results['documents'][0]):
    distance = results['distances'][0][i]
    print(f"[อันดับที่ {i+1}] (Distance score: {distance:.4f}):\n{doc}\n{'-'*40}")

## 7. ตอบคำถามผู้ใช้ด้วยโมเดล Gemini (Augmented Generation)

นำข้อความที่ดึงมาจากฐานข้อมูลเวกเตอร์ข้างต้น ไปประกอบร่างต่อกับคำถามเดิมของผู้ใช้ในรูปแบบ Prompt (เราเรียกว่า Context-augmented Prompt) แล้วป้อนให้โมเดล `gemini-1.5-flash` ช่วยตอบกลับมา

In [ ]:
# รวมบริบทจากผลการค้นหา
contexts = results['documents'][0]
combined_context = "\n---\n".join(contexts)

# สร้าง Prompt ครบวงจร
system_instruction = """คุณเป็นเจ้าหน้าที่ HR ตอบคำถามพนักงานช่วยเหลือในระบบสอบถามข้อมูลอัตโนมัติ
ให้ใช้เฉพาะข้อมูลที่อยู่ในระบบอ้างอิง (Context) ที่ให้ไว้เท่านั้นในการตอบคำถามอย่างถูกต้อง สุภาพ และอิงอยู่บนความจริง
หากคำตอบไม่ได้อยู่ในเอกสารอ้างอิง ให้บอกตรงๆ ว่า \"ข้อมูลไม่ปรากฏในสวัสดิการปัจจุบัน กรุณาติดต่อแผนก HR เพิ่มเติม\" ห้ามเดาข้อมูล"""

user_prompt = f"""
เอกสารอ้างอิง:
{combined_context}

คำถามของพนักงาน: {query_text}
"""

# เรียก Gemini LLM
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=user_prompt,
    config=types.GenerateContentConfig(system_instruction=system_instruction)
)

print("=== คำตอบจาก HR AI Assistant ===\n")
print(response.text)

## สรุปบทเรียน

จากโค้ดข้างต้น ทุกคนจะเห็นแล้วว่ากระบวนการ RAG ประกอบไปด้วยการสกัด การแบ่งข้อความ ยัดลงฐานข้อมูลเวกเตอร์ และการส่งข้อมูลเข้าไปแนบกับคำถามของโมเดล

ในบทเรียนถัดไป (Session 3) เราจะพัฒนาโฟลว์เหล่านี้ให้ทำงานโดยอัตโนมัติบน **Apache Airflow 3.x** เมื่อมีไฟล์เอกสารเข้ามาวางในระบบท่อข้อมูลข้อมูล!